In [ ]:
# ============================================================
# Table 8 ONLY: Modern attack–defense replication on FEMNIST
#   - Dataset: FEMNIST via HuggingFace/Flower Datasets (no LEAF)
#   - Attacks: PoisonedFL-style, Backdoor/Model Replacement
#   - Defenses: FedCC (linear CKA filter), Attack-Adaptive Aggregation
#   - Output: prints ONLY Table 8 (M_head, W, gap, PoG), tail-mean over last K rounds
#
# Install once (Colab):
#   pip -q install flwr-datasets[vision] datasets
# ============================================================

import gc
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset


# ============================================================
# 0) Utilities
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def cleanup() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


@torch.no_grad()
def accuracy(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return float(correct / total) if total > 0 else 0.0


def tail_mean(vals: List[float], K: int) -> float:
    if len(vals) == 0:
        return float("nan")
    xs = vals[-K:] if len(vals) > K else vals
    return float(np.mean(xs))


# ============================================================
# 1) Paper-aligned config for Table 8
# ============================================================

@dataclass
class Table8Config:
    # FL budget (paper: FedAvg for 100 rounds, partial participation) :contentReference[oaicite:2]{index=2}
    N: int = 32                 # number of clients
    T: int = 100                # rounds
    E: int = 1                  # local epochs
    B: int = 64                 # batch size
    eta: float = 0.02           # lr
    mu: float = 0.9             # momentum
    wd: float = 0.0             # weight decay

    m: int = 12                 # clients per round (partial participation)

    # FEMNIST: digits=head, letters=tail :contentReference[oaicite:3]{index=3}
    head_classes: Tuple[int, ...] = tuple(range(0, 10))
    tail_classes: Tuple[int, ...] = tuple(range(10, 62))

    # public sets (CKA support + head/tail eval)
    support_size: int = 128
    head_eval_size: int = 4096
    tail_eval_size: int = 4096

    # malicious + schedule
    theta_mal: float = 0.30
    t0: int = 5

    # PoisonedFL-ish knobs
    scale_poison: float = 2.2
    scale_min: float = 1.0
    scale_max: float = 4.0
    dyn_eta: float = 0.8
    beta_consistency: float = 1e-3
    mu_global: float = 5e-4
    flip_tail_labels: bool = True

    # Backdoor / model replacement knobs
    backdoor_target: int = 0
    backdoor_poison_frac: float = 0.30
    model_replacement_gamma: float = 6.0
    trigger_size: int = 3
    trigger_value: float = 1.0  # clamped to [-1,1]

    # FedCC (CKA) filter
    fedcc_reject_frac: float = 0.25
    fedcc_min_keep: int = 3

    # Attack-adaptive aggregation
    adaagg_alpha: float = 3.0

    # table summary: last K rounds tail-mean :contentReference[oaicite:4]{index=4}
    K_tail: int = 10

    # client filtering when building writers
    min_train: int = 40
    min_test: int = 10
    max_try: int = 5000

    seed: int = 42
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


CFG = Table8Config()
set_seed(CFG.seed)


# ============================================================
# 2) FEMNIST via flwr_datasets (writer_id partitions)
# ============================================================

def build_femnist_clients_from_hf(
    n_clients: int,
    train_ratio: float,
    seed: int,
    min_train: int,
    min_test: int,
    max_try: int,
) -> Tuple[List[TensorDataset], List[TensorDataset], int]:
    """
    Collects first n_clients valid writer partitions.
    Returns client_train_sets, client_test_sets, num_classes (inferred).
    """
    from flwr_datasets import FederatedDataset
    from flwr_datasets.partitioner import NaturalIdPartitioner

    rng = np.random.default_rng(seed)
    fds = FederatedDataset(
        dataset="flwrlabs/femnist",
        partitioners={"train": NaturalIdPartitioner(partition_by="writer_id")},
    )

    client_train_sets: List[TensorDataset] = []
    client_test_sets: List[TensorDataset] = []
    max_label = -1

    def to_tensors(part) -> Tuple[torch.Tensor, torch.Tensor]:
        images = part["image"]  # PIL
        labels = np.array(part["character"], dtype=np.int64)
        X = np.stack([np.array(img, dtype=np.float32) for img in images], axis=0)  # [N,28,28]
        X = X / 255.0
        X = (X - 0.5) / 0.5
        X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # [N,1,28,28]
        Y = torch.tensor(labels, dtype=torch.long)
        return X, Y

    for pid in range(max_try):
        if len(client_train_sets) >= n_clients:
            break
        try:
            part = fds.load_partition(partition_id=pid, split="train")
        except Exception:
            break

        X, Y = to_tensors(part)
        n = len(Y)
        if n < (min_train + min_test):
            continue

        idx = np.arange(n)
        rng.shuffle(idx)
        cut = int(round(train_ratio * n))
        tr_idx, te_idx = idx[:cut], idx[cut:]

        if len(tr_idx) < min_train or len(te_idx) < min_test:
            continue

        client_train_sets.append(TensorDataset(X[tr_idx], Y[tr_idx]))
        client_test_sets.append(TensorDataset(X[te_idx], Y[te_idx]))
        max_label = max(max_label, int(Y.max().item()))

    if len(client_train_sets) < n_clients:
        raise RuntimeError(f"Not enough FEMNIST clients: {len(client_train_sets)}/{n_clients}")

    return client_train_sets, client_test_sets, max_label + 1


def filter_by_labels(ds: TensorDataset, allowed: Tuple[int, ...]) -> TensorDataset:
    X, Y = ds.tensors
    if len(Y) == 0:
        return TensorDataset(X, Y)
    allowed_set = set(int(a) for a in allowed)
    mask = torch.zeros_like(Y, dtype=torch.bool)
    for c in allowed_set:
        mask |= (Y == c)
    return TensorDataset(X[mask], Y[mask])


def sample_public_from_clients(
    client_train_sets: List[TensorDataset],
    allowed_labels: Tuple[int, ...],
    total_size: int,
    rng: np.random.Generator,
) -> TensorDataset:
    xs, ys = [], []
    for ds in client_train_sets:
        ds_f = filter_by_labels(ds, allowed_labels)
        if len(ds_f) == 0:
            continue
        X, Y = ds_f.tensors
        xs.append(X)
        ys.append(Y)
    if len(xs) == 0:
        return TensorDataset(torch.empty(0, 1, 28, 28), torch.empty(0, dtype=torch.long))
    X = torch.cat(xs, dim=0)
    Y = torch.cat(ys, dim=0)
    n = len(Y)
    idx = np.arange(n)
    rng.shuffle(idx)
    idx = idx[: min(total_size, n)]
    return TensorDataset(X[idx], Y[idx])


# ============================================================
# 3) Model (with penultimate features for CKA)
# ============================================================

class SimpleCNNFeat(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x, return_feat: bool = False):
        z = self.conv(x).flatten(1)
        feat = self.relu(self.fc1(z))
        logits = self.fc2(feat)
        if return_feat:
            return logits, feat
        return logits


def make_model(num_classes: int, device: torch.device) -> nn.Module:
    return SimpleCNNFeat(num_classes=num_classes).to(device)


@torch.no_grad()
def collect_features(model: nn.Module, loader: DataLoader, device: torch.device) -> torch.Tensor:
    model.eval()
    feats = []
    for x, _ in loader:
        x = x.to(device, non_blocking=True)
        _, f = model(x, return_feat=True)
        feats.append(f.detach())
    if len(feats) == 0:
        return torch.empty(0, 128, device=device)
    return torch.cat(feats, dim=0)


# ============================================================
# 4) Local training (honest / PoisonedFL-ish)
# ============================================================

def local_train_honest(model: nn.Module, dataset, cfg: Table8Config) -> Dict[str, torch.Tensor]:
    loader = DataLoader(dataset, batch_size=cfg.B, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
    opt = optim.SGD(model.parameters(), lr=cfg.eta, momentum=cfg.mu, weight_decay=cfg.wd)
    crit = nn.CrossEntropyLoss()

    model.train()
    for _ in range(cfg.E):
        for x, y in loader:
            x = x.to(cfg.device, non_blocking=True)
            y = y.to(cfg.device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(x), y)
            loss.backward()
            opt.step()

    return {k: v.detach().clone() for k, v in model.state_dict().items()}


def _sd_to_device(sd_cpu: Dict[str, torch.Tensor], device: torch.device) -> Dict[str, torch.Tensor]:
    return {k: v.to(device) for k, v in sd_cpu.items()}


def _param_l2(sd_a: Dict[str, torch.Tensor], sd_b: Dict[str, torch.Tensor], device: torch.device) -> torch.Tensor:
    s = torch.tensor(0.0, device=device)
    for k in sd_a.keys():
        s = s + (sd_a[k] - sd_b[k]).pow(2).sum()
    return s


def local_train_poisonedfl(
    model: nn.Module,
    dataset,
    global_sd_cpu: Dict[str, torch.Tensor],
    prev_mal_tx_sd_cpu: Optional[Dict[str, torch.Tensor]],
    cfg: Table8Config,
) -> Dict[str, torch.Tensor]:
    """
    loss = CE(y_mod) + mu_global * ||w-w_global||^2 + beta * ||w-w_prev_mal||^2
    """
    loader = DataLoader(dataset, batch_size=cfg.B, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
    opt = optim.SGD(model.parameters(), lr=cfg.eta, momentum=cfg.mu, weight_decay=cfg.wd)
    crit = nn.CrossEntropyLoss()

    global_sd = _sd_to_device(global_sd_cpu, cfg.device)
    prev_sd = _sd_to_device(prev_mal_tx_sd_cpu, cfg.device) if prev_mal_tx_sd_cpu is not None else None

    head_arr = torch.tensor(list(cfg.head_classes), device=cfg.device, dtype=torch.long)
    tail_set = set(int(x) for x in cfg.tail_classes)

    model.train()
    for _ in range(cfg.E):
        for x, y in loader:
            x = x.to(cfg.device, non_blocking=True)
            y = y.to(cfg.device, non_blocking=True)

            y_mod = y
            if cfg.flip_tail_labels:
                y_mod = y.clone()
                tail_mask = torch.zeros_like(y_mod, dtype=torch.bool)
                for c in tail_set:
                    tail_mask |= (y_mod == c)
                if tail_mask.any() and len(head_arr) > 0:
                    ridx = torch.randint(0, len(head_arr), (tail_mask.sum().item(),), device=cfg.device)
                    y_mod[tail_mask] = head_arr[ridx]

            logits = model(x)
            loss_ce = crit(logits, y_mod)

            cur_sd = {k: v for k, v in model.state_dict().items()}
            loss_glob = _param_l2(cur_sd, global_sd, cfg.device)

            if prev_sd is not None:
                loss_cons = _param_l2(cur_sd, prev_sd, cfg.device)
            else:
                loss_cons = torch.tensor(0.0, device=cfg.device)

            loss = loss_ce + cfg.mu_global * loss_glob + cfg.beta_consistency * loss_cons

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

    return {k: v.detach().clone() for k, v in model.state_dict().items()}


# ============================================================
# 5) Backdoor dataset wrapper
# ============================================================

class BackdoorWrapper(Dataset):
    def __init__(self, base: TensorDataset, poison_frac: float, target_label: int,
                 trigger_size: int, trigger_value: float, seed: int):
        self.base = base
        self.poison_frac = float(poison_frac)
        self.target_label = int(target_label)
        self.trigger_size = int(trigger_size)
        self.trigger_value = float(trigger_value)
        rng = np.random.default_rng(seed)

        n = len(base)
        idx = np.arange(n)
        rng.shuffle(idx)
        self.poison_idx = set(idx[: int(round(self.poison_frac * n))].tolist())

    def __len__(self):
        return len(self.base)

    def _apply_trigger(self, x: torch.Tensor) -> torch.Tensor:
        x2 = x.clone()
        s = self.trigger_size
        v = torch.tensor(self.trigger_value, dtype=x2.dtype)
        x2[:, -s:, -s:] = torch.clamp(v, -1.0, 1.0)
        return x2

    def __getitem__(self, idx: int):
        x, y = self.base[idx]
        if idx in self.poison_idx:
            x = self._apply_trigger(x)
            y = torch.tensor(self.target_label, dtype=torch.long)
        return x, y


# ============================================================
# 6) State helpers + defenses (FedCC / Attack-Adaptive Aggregation)
# ============================================================

def sd_to_cpu(sd: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    return {k: v.detach().to("cpu").clone() for k, v in sd.items()}


def make_tx_state_cpu(global_sd_cpu: Dict[str, torch.Tensor], local_sd: Dict[str, torch.Tensor], scale: float) -> Dict[str, torch.Tensor]:
    local_sd_cpu = sd_to_cpu(local_sd)
    tx = {}
    for k in global_sd_cpu.keys():
        tx[k] = global_sd_cpu[k] + float(scale) * (local_sd_cpu[k] - global_sd_cpu[k])
    return tx


def fedavg_mean_cpu(states_cpu: List[Dict[str, torch.Tensor]], weights: Optional[List[float]] = None) -> Dict[str, torch.Tensor]:
    if weights is None:
        weights = [1.0] * len(states_cpu)
    wsum = float(sum(weights))
    out = {}
    keys = states_cpu[0].keys()
    for k in keys:
        acc = None
        for sd, w in zip(states_cpu, weights):
            term = sd[k] * float(w)
            acc = term if acc is None else (acc + term)
        out[k] = acc / wsum
    return out


def flatten_update_cpu(global_sd_cpu: Dict[str, torch.Tensor], tx_sd_cpu: Dict[str, torch.Tensor]) -> torch.Tensor:
    vecs = []
    for k in global_sd_cpu.keys():
        vecs.append((tx_sd_cpu[k] - global_sd_cpu[k]).reshape(-1))
    return torch.cat(vecs, dim=0)  # CPU tensor


def linear_cka(X: torch.Tensor, Y: torch.Tensor, eps: float = 1e-12) -> float:
    if X.numel() == 0 or Y.numel() == 0:
        return 0.0
    n = min(X.shape[0], Y.shape[0])
    X = X[:n]
    Y = Y[:n]
    Xc = X - X.mean(dim=0, keepdim=True)
    Yc = Y - Y.mean(dim=0, keepdim=True)
    XT_Y = Xc.t() @ Yc
    XT_X = Xc.t() @ Xc
    YT_Y = Yc.t() @ Yc
    num = (XT_Y.pow(2)).sum()
    den = torch.sqrt((XT_X.pow(2)).sum() * (YT_Y.pow(2)).sum() + eps)
    return float((num / (den + eps)).clamp(0.0, 1.0).item())


def defense_fedcc(global_sd_cpu: Dict[str, torch.Tensor], client_tx_cpu: List[Dict[str, torch.Tensor]],
                  support_loader: DataLoader, num_classes: int, cfg: Table8Config) -> Tuple[Dict[str, torch.Tensor], float]:
    """
    FedCC: compute linear CKA similarity between global and each client model on public support;
    reject lowest fraction and FedAvg remaining.
    Returns (new_global_sd_cpu, reject_rate).
    """
    g_model = make_model(num_classes, cfg.device)
    g_model.load_state_dict(global_sd_cpu, strict=True)
    G = collect_features(g_model, support_loader, cfg.device)
    del g_model
    cleanup()

    sims = []
    for tx in client_tx_cpu:
        c_model = make_model(num_classes, cfg.device)
        c_model.load_state_dict(tx, strict=True)
        C = collect_features(c_model, support_loader, cfg.device)
        sims.append(linear_cka(G, C))
        del c_model, C
        cleanup()

    n = len(client_tx_cpu)
    reject_n = int(round(cfg.fedcc_reject_frac * n))
    reject_n = min(reject_n, max(0, n - cfg.fedcc_min_keep))

    order = np.argsort(np.array(sims))  # ascending similarity
    keep_idx = order[reject_n:].tolist()
    kept = [client_tx_cpu[i] for i in keep_idx]

    new_sd = fedavg_mean_cpu(kept)
    reject_rate = float(reject_n / max(1, n))

    del G, sims, kept
    cleanup()
    return new_sd, reject_rate


def defense_attack_adaptive_agg(global_sd_cpu: Dict[str, torch.Tensor], client_tx_cpu: List[Dict[str, torch.Tensor]], cfg: Table8Config) -> Dict[str, torch.Tensor]:
    """
    Attack-adaptive aggregation: downweight suspicious updates using norm z-score + cosine distance to median update.
    """
    updates = [flatten_update_cpu(global_sd_cpu, tx) for tx in client_tx_cpu]
    U = torch.stack(updates, dim=0)  # [n, d] on CPU

    med = U.median(dim=0).values
    med_norm = med.norm() + 1e-12

    norms = torch.tensor([u.norm().item() for u in updates], dtype=torch.float32)
    z = (norms - norms.mean()) / (norms.std() + 1e-12)

    cos_terms = []
    for u in updates:
        cos_terms.append((u @ med) / (u.norm() * med_norm + 1e-12))
    cos_terms = torch.stack(cos_terms, dim=0).clamp(-1.0, 1.0)
    cos_dist = 1.0 - cos_terms

    score = z + cos_dist
    weights = torch.softmax(-cfg.adaagg_alpha * score, dim=0).numpy().tolist()

    new_sd = fedavg_mean_cpu(client_tx_cpu, weights=weights)

    del updates, U, med, norms, z, cos_terms, cos_dist, score
    cleanup()
    return new_sd


# ============================================================
# 7) Run one condition -> Table 8 summary stats (tail-mean over last K rounds)
# ============================================================

def run_one_condition(
    cfg: Table8Config,
    num_classes: int,
    client_train_sets: List[TensorDataset],
    client_tail_test_loaders: List[Optional[DataLoader]],
    support_loader: DataLoader,
    head_eval_loader: DataLoader,
    tail_eval_loader: DataLoader,
    defense: str,   # "FedCC" | "AttackAdaptiveAggregation"
    attack: str,    # "NONE" | "POISONEDFL" | "BACKDOOR"
    seed_offset: int,
) -> Dict[str, float]:
    """
    Returns dict with: M_head, W, gap (tail-mean over last K rounds)
    (PoG is computed outside relative to same-defense baseline) :contentReference[oaicite:5]{index=5}
    """
    rng = np.random.default_rng(cfg.seed + seed_offset)

    N = len(client_train_sets)
    n_mal = int(round(cfg.theta_mal * N))
    ids = np.arange(N)
    rng.shuffle(ids)
    mal_ids = set(ids[:n_mal].tolist())

    # build per-client training dataset (backdoor wraps malicious)
    built_train = []
    for cid in range(N):
        base = client_train_sets[cid]
        if (attack == "BACKDOOR") and (cid in mal_ids):
            built_train.append(
                BackdoorWrapper(
                    base=base,
                    poison_frac=cfg.backdoor_poison_frac,
                    target_label=cfg.backdoor_target,
                    trigger_size=cfg.trigger_size,
                    trigger_value=cfg.trigger_value,
                    seed=int(cfg.seed + 777 + seed_offset + cid),
                )
            )
        else:
            built_train.append(base)

    # init global state on CPU
    global_model = make_model(num_classes, cfg.device)
    global_sd_cpu = sd_to_cpu(global_model.state_dict())
    del global_model
    cleanup()

    # PoisonedFL memory (store last transmitted state per malicious client; CPU)
    prev_mal_tx_cpu: Dict[int, Optional[Dict[str, torch.Tensor]]] = {cid: None for cid in mal_ids}
    last_reject_rate = 0.0

    M_head_hist: List[float] = []
    W_hist: List[float] = []
    gap_hist: List[float] = []

    for t in range(cfg.T):
        part = rng.choice(N, size=min(cfg.m, N), replace=False).tolist()
        client_tx_cpu: List[Dict[str, torch.Tensor]] = []

        for cid in part:
            local_model = make_model(num_classes, cfg.device)
            local_model.load_state_dict(global_sd_cpu, strict=True)

            is_active = (attack != "NONE") and (t >= cfg.t0) and (cid in mal_ids)

            if is_active and attack == "POISONEDFL":
                local_sd_dev = local_train_poisonedfl(
                    model=local_model,
                    dataset=built_train[cid],
                    global_sd_cpu=global_sd_cpu,
                    prev_mal_tx_sd_cpu=prev_mal_tx_cpu[cid],
                    cfg=cfg,
                )

                # dynamic magnitude tweak (only meaningful for FedCC)
                scale = cfg.scale_poison * (1.0 + cfg.dyn_eta * (last_reject_rate - cfg.fedcc_reject_frac))
                scale = float(np.clip(scale, cfg.scale_min, cfg.scale_max))

                tx_cpu = make_tx_state_cpu(global_sd_cpu, local_sd_dev, scale=scale)
                client_tx_cpu.append(tx_cpu)
                prev_mal_tx_cpu[cid] = {k: v.clone() for k, v in tx_cpu.items()}

                del local_sd_dev

            else:
                local_sd_dev = local_train_honest(local_model, built_train[cid], cfg)

                # backdoor/model replacement: scale up malicious updates
                if is_active and attack == "BACKDOOR":
                    scale = cfg.model_replacement_gamma
                else:
                    scale = 1.0

                tx_cpu = make_tx_state_cpu(global_sd_cpu, local_sd_dev, scale=scale)
                client_tx_cpu.append(tx_cpu)

                del local_sd_dev

            del local_model
            cleanup()

        # defense / aggregation
        if defense == "FedCC":
            new_sd_cpu, rej_rate = defense_fedcc(
                global_sd_cpu, client_tx_cpu, support_loader, num_classes, cfg
            )
            last_reject_rate = rej_rate
            global_sd_cpu = new_sd_cpu
        elif defense == "AttackAdaptiveAggregation":
            global_sd_cpu = defense_attack_adaptive_agg(global_sd_cpu, client_tx_cpu, cfg)
            last_reject_rate = 0.0
        else:
            raise ValueError(defense)

        del client_tx_cpu
        cleanup()

        # evaluate: M_head (public head metric), W (tail welfare across clients), gap=M_head-W :contentReference[oaicite:6]{index=6}
        eval_model = make_model(num_classes, cfg.device)
        eval_model.load_state_dict(global_sd_cpu, strict=True)

        M_head = accuracy(eval_model, head_eval_loader, cfg.device)

        tail_accs = []
        for loader in client_tail_test_loaders:
            if loader is None:
                continue
            tail_accs.append(accuracy(eval_model, loader, cfg.device))
        W = float(np.mean(tail_accs)) if len(tail_accs) > 0 else 0.0

        gap = float(M_head - W)

        M_head_hist.append(M_head)
        W_hist.append(W)
        gap_hist.append(gap)

        del eval_model
        cleanup()

    # tail-mean over last K rounds
    M_head_bar = tail_mean(M_head_hist, cfg.K_tail)
    W_bar = tail_mean(W_hist, cfg.K_tail)
    gap_bar = tail_mean(gap_hist, cfg.K_tail)

    del global_sd_cpu, built_train, prev_mal_tx_cpu, M_head_hist, W_hist, gap_hist
    cleanup()

    return {"M_head": M_head_bar, "W": W_bar, "gap": gap_bar}


# ============================================================
# 8) Table 8 driver
# ============================================================

def run_table_8(cfg: Table8Config) -> pd.DataFrame:
    rng = np.random.default_rng(cfg.seed)

    # load clients
    client_train_sets, client_test_sets, num_classes = build_femnist_clients_from_hf(
        n_clients=cfg.N,
        train_ratio=0.8,
        seed=cfg.seed,
        min_train=cfg.min_train,
        min_test=cfg.min_test,
        max_try=cfg.max_try,
    )

    # public sets
    public_head = sample_public_from_clients(client_train_sets, cfg.head_classes, cfg.head_eval_size, rng)
    public_tail = sample_public_from_clients(client_train_sets, cfg.tail_classes, cfg.tail_eval_size, rng)
    public_support = sample_public_from_clients(client_train_sets, cfg.head_classes, cfg.support_size, rng)
    if len(public_support) == 0:
        public_support = sample_public_from_clients(client_train_sets, tuple(range(num_classes)), cfg.support_size, rng)

    head_eval_loader = DataLoader(public_head, batch_size=cfg.B, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
    tail_eval_loader = DataLoader(public_tail, batch_size=cfg.B, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
    support_loader = DataLoader(public_support, batch_size=cfg.B, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

    # client tail test loaders (welfare)
    client_tail_test_loaders: List[Optional[DataLoader]] = []
    for ds in client_test_sets:
        ds_tail = filter_by_labels(ds, cfg.tail_classes)
        if len(ds_tail) == 0:
            client_tail_test_loaders.append(None)
        else:
            client_tail_test_loaders.append(
                DataLoader(ds_tail, batch_size=cfg.B, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
            )

    defenses = ["FedCC", "AttackAdaptiveAggregation"]
    attacks = ["NONE", "POISONEDFL", "BACKDOOR"]

    # run and collect
    results = {(atk, dfn): None for atk in attacks for dfn in defenses}

    seed_offset = 0
    for dfn in defenses:
        for atk in attacks:
            results[(atk, dfn)] = run_one_condition(
                cfg=cfg,
                num_classes=num_classes,
                client_train_sets=client_train_sets,
                client_tail_test_loaders=client_tail_test_loaders,
                support_loader=support_loader,
                head_eval_loader=head_eval_loader,
                tail_eval_loader=tail_eval_loader,
                defense=dfn,
                attack=atk,
                seed_offset=seed_offset,
            )
            seed_offset += 1

    # baseline W per defense
    W_base = {dfn: results[("NONE", dfn)]["W"] for dfn in defenses}

    # build Table 8 rows (exact columns in paper) :contentReference[oaicite:7]{index=7}
    def attack_label(a: str) -> str:
        if a == "NONE":
            return "None"
        if a == "POISONEDFL":
            return "PoisonedFL"
        if a == "BACKDOOR":
            return "Backdoor/Model Rep."
        return a

    rows = []
    for atk in attacks:
        row = {"Attack": attack_label(atk)}
        for dfn in defenses:
            M = results[(atk, dfn)]["M_head"]
            W = results[(atk, dfn)]["W"]
            gap = results[(atk, dfn)]["gap"]

            if atk == "NONE":
                pog = np.nan
            else:
                wb = W_base[dfn]
                pog = np.nan if (not np.isfinite(wb) or wb <= 1e-12) else (wb - W) / wb

            prefix = "FedCC" if dfn == "FedCC" else "Attack-Adaptive Aggregation"
            row[f"{prefix} M_head"] = M
            row[f"{prefix} W"] = W
            row[f"{prefix} gap"] = gap
            row[f"{prefix} PoG"] = pog
        rows.append(row)

    df = pd.DataFrame(rows)

    # formatting like the paper (4 decimals; PoG baseline shown as "–")
    def fmt(x):
        if pd.isna(x):
            return "–"
        return f"{x:.4f}"

    for c in df.columns:
        if c == "Attack":
            continue
        df[c] = df[c].map(fmt)

    # cleanup big objects
    del client_train_sets, client_test_sets, client_tail_test_loaders
    cleanup()

    return df


if __name__ == "__main__":
    df8 = run_table_8(CFG)
    print("Table 8: Modern attack–defense replication on FEMNIST (tail-mean over last K rounds).")
    print(df8.to_string(index=False))